1. Problem Statement
2. Objective
3. Setting up the LLM (Groq)
4. SQL Agent for Data Retrieval
5. Search Agent using DuckDuckGo
6. Interface Between SQL and Search Agent
7. Filtering Trusted News Sources
8. Output Generation using LLM
9. Querying the Agent
10. Conclusion

## Problem Statement

NewsFindr is focused on improving how users discover news by providing real-time updates based on individual interests. Currently, most platforms either show generic feeds or require manual searching, which often leads to information overload and makes it difficult to find relevant and trustworthy content.

As part of this project, we are building an AI-powered news retrieval system using an agent-based approach. The goal is to create a system that can fetch personalized, up-to-date news while ensuring that the information comes from credible sources.

## Objective

The objective of this project is to design a structured, multi-step news retrieval system that:

- Provides real-time news updates based on user interests
- Ensures accuracy by selecting trusted sources
- Reduces information overload by filtering and summarizing content
- Improves user experience by delivering relevant and easy-to-understand news

This system uses an agent-based workflow to make the process more efficient and reliable.

## Importing Libraries

In [90]:
%pip install groq
%pip install ddgs

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Setting Up LLM

In this step, we set up the Groq LLM, which will be used later for generating summaries of the news content.

Before integrating it into the pipeline, we tested it with a simple query to confirm that it is working correctly and generating meaningful responses.

In [91]:
# LLM Model to be used for summarization
LLM_MODEL = "llama-3.3-70b-versatile"

In [92]:
from groq import Groq
import os  

#intialize the Groq client
#Created an environment variable named GROQ_API_KEY and set it to Groq API key before running this code.
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

#Test the connection by making a simple request to Groq LLM 
response = client.chat.completions.create(
     model= LLM_MODEL,
     messages=[{"role": "user", "content": """'Expain Groq in one line'"""}]
)


print(response.choices[0].message.content)

Groq is an open-source, software-based, AI-specific processor designed by Google for machine learning workloads.


## SQL Agent for Data Retrieval

To personalize the news results, a SQL-based agent has been created that stores user information such as email and their area of interest.

When a user provides their email, the system verifies it and retrieves their preferred category. This ensures that the news recommendations are tailored specifically to each user, making the system more relevant and user-focused.

In [93]:
import sqlite3

conn = sqlite3.connect('customer.db')
cursor = conn.cursor()

Function

In [94]:
# verify if the user exists in the database and return their interest
def verify_user(email):
    cursor.execute("SELECT * FROM customers WHERE email=?", (email,))
    user = cursor.fetchone()
    
    if user:
        print("User verified:", email)
        return True
    else:
        print("User not found", email)
        return False

In [95]:
# get the user's interest from the database
def get_user_interest(email):

    if not verify_user(email):
        return "Access denied"
    cursor.execute("SELECT interests FROM customers WHERE email=?", (email,))
    result = cursor.fetchone()
    return result[0] if result else None

## Search Agent using DuckDuckGo


In this step, a search agent has been implemented using DuckDuckGo to fetch real-time news articles.

The system generates a search query based on the user’s interest and retrieves the latest news results. This helps ensure that the information provided is current and not outdated.

In [96]:
from ddgs import DDGS
# search for news articles based on the user's interest using DuckDuckGo Search API
def search_news(query):
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(r)
    return results

## Interface Between SQL and Search Agent


I structured this step separately to clearly define how user interest is converted into a search query before passing it to the search agent.



This step connects the SQL agent and the search agent.

Once the user’s interest is retrieved from the database, it is then converted into a structured search query. This helps in fetching more precise and relevant results.

For example, if the user is interested in "finance", the system generates a query like "latest finance news 2026".

In [97]:
# generate a search query based on the user's interest
def generate_search_query(interest):
    return f"{interest} latest news article 2026 site:reuters.com OR site:bbc.com OR site:cnn.com"

## Filtering Trusted News Sources


To maintain credibility, we added a filtering step where only trusted news sources are selected.

This ensures that the system avoids unreliable or misleading content and focuses only on well-known and credible platforms such as BBC, Reuters, CNN, and NDTV.

This step is important for ensuring the accuracy and reliability of the recommendations.

In [98]:
# filter the search results to include only news articles from trusted sources and exclude homepage or category pages
def filter_trusted_news(results):
    trusted_sources = ["reuters.com", "bbc.com", "cnn.com", "ndtv.com"]
    
    filtered = []
    for r in results:
        url = r.get("href", "").lower()
        
        # Check trusted source
        if any(src in url for src in trusted_sources):
            
            # Reject homepage / category pages
            if not url.endswith((".com/", ".com")) and len(url.split("/")) > 4:
                filtered.append(r)
    
    return filtered

## Output Generation using LLM


This step ensures that users do not need to read full articles and can quickly understand the key information.

After retrieving and filtering the news articles, we have used the LLM to generate summaries.

Instead of showing full articles, the system provides short summaries so users can quickly understand the key points. This improves readability and helps users save time while going through multiple news items.

In [99]:
# summarize the news article using Groq LLM
def summarize_news(text):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": f"Summarize this news: {text}"}]
    )
    return response.choices[0].message.content

## Final Agent

In [100]:
# to validate if the article is a news article and not a homepage or category page
def is_valid_article(text):
    if not text:
        return False
    if "visit" in text.lower() or "latest news" in text.lower():
        return False
    return True

In [101]:
# Main Agent Function
def news_agent(email):
    interest = get_user_interest(email)
    final_output = []
    if interest is None or interest == "Access denied":
        final_output.append("User interest not found or access denied.")
        return final_output
    
    query = generate_search_query(interest)
    results = search_news(query)
    filtered_results = filter_trusted_news(results)
    
    # If less than 3 valid articles, return all
    for i, r in enumerate(filtered_results[:3], 1):
        content = r.get("body", "")
        
        summary = summarize_news(content)
        
        news_item = {
            "title": r["title"],
            "url": r["href"],
            "summary": summary
        }
        
        final_output.append(news_item)

        # Print nicely
        print(f"\nNews {i}")
        print("Title:", r["title"])
        print("URL:", r["href"])
        print("Summary:", summary)
        print("-" * 60)

    return final_output

## Querying the Agent


To test the system, we ran multiple queries using different user emails.

For each query, the system:

- Retrieves the user’s interest
- Generates a search query
- Fetches real-time news
- Filters trusted sources
- Generates summaries

This demonstrates that the system is working end-to-end and providing personalized news results.

In [102]:
# News Agent Scenarios
news_agent("kevin.f8641860-7@gmail.com")
news_agent("alice.6eb33c45-5@gmail.com")
news_agent("ian.aee37571-7@gmail.com")
news_agent("invalid@gmail.com") #Negative Scenario

User verified: kevin.f8641860-7@gmail.com

News 1
Title: How AI and politics dominated Davos | Reuters
URL: https://www.reuters.com/technology/artificial-intelligence/artificial-intelligencer-how-ai-politics-dominated-davos-2026-01-22/
Summary: The news mentions a person who joined Reuters in 2014, initially covering airlines and travel from New York, and has since moved on to focus on startups, tech investments, and AI.
------------------------------------------------------------

News 2
Title: March 19, 2026 - Sitemap | Reuters
URL: https://www.reuters.com/sitemap/2026-03/19/1/
Summary: The Palestinian national team has canceled their friendly matches in Morocco due to travel disruptions in the Middle East.
------------------------------------------------------------

News 3
Title: March 18, 2026 - Sitemap | Reuters
URL: https://www.reuters.com/sitemap/2026-03/18/1/
Summary: The German foreign ministry has issued a travel warning, strongly advising against traveling to Cuba.
--------

['User interest not found or access denied.']

## Conclusion

This structured approach also makes the system more explainable and easier to scale in real-world applications.

In this project, we implemented an agent-based news retrieval system aligned with the NewsFindr problem statement.

By combining SQL for user personalization, DuckDuckGo for real-time search, and Groq LLM for summarization, the system delivers relevant, timely, and credible news content.

This approach helps reduce information overload and improves how users interact with news platforms. The modular design also makes the system scalable and easy to enhance in the future.